# Paper Figures: Figure 3 - Cluster Analysis

This notebook generates publication-ready figures for cluster analysis of dopamine responses using data assembled by `src/assemble_all_data.py`.

**Figure 3: Cluster Analysis** — Pie charts showing cluster composition, representative heatmaps by rat, and summary cluster responses with statistical analysis.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Register dill/pathlib compatibility shim BEFORE importing dill
sys.path.insert(0, str(Path("../src").resolve()))
from pickle_compat import enable_dill_pathlib_compat
enable_dill_pathlib_compat()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import dill
from scipy import stats

from figure_config import (
    configure_matplotlib, COLORS,
    DATAFOLDER, RESULTSFOLDER, FIGSFOLDER,
    HEATMAP_CMAP_DIV,
    SAVE_FIGS
)
from figure_plotting import scale_vlim_to_data

from trompy import save_figure_atomic

# Configure matplotlib
configure_matplotlib()
colors = COLORS  # Use shared color palette
custom_cmap = HEATMAP_CMAP_DIV  # Use shared colormap

## Load Assembled Data

Load the complete dataset from the pickle file generated by the assembly script.

In [ ]:
assembled_data_path = DATAFOLDER / "assembled_data.pickle"

with open(assembled_data_path, "rb") as f:
    data = dill.load(f)

# Extract main components
x_array = data["x_array"]
snips_photo = data["snips_photo"]
snips_behav = data["snips_simba"]
fits_df = data["fits_df"]
metadata = data.get("metadata", {})

print(f"Loaded assembled data from {assembled_data_path}")
print(f"\nData structure:")
print(f"  - x_array shape: {x_array.shape}")
print(f"  - snips_photo shape: {snips_photo.shape}")
print(f"  - x_array columns: {x_array.columns.tolist()}")
print(f"  - Number of trials: {len(x_array)}") 

# Check data processing metadata
print(f"\nData processing metadata:")
if metadata:
    print(f"  Behaviour metric: {metadata.get('behav_metric', 'unknown')}")
    print(f"  Photometry smoothed: {metadata.get('photo_smoothed', False)}")
    print(f"  Photometry z-scored: {metadata.get('photo_zscored', False)}")
else:
    print(f"  No metadata found; assuming standard processing")

## Figure 3: Cluster Analysis — Dopamine Response Clustering

Analysis of dopamine neuron clustering during sodium appetite showing pie charts of cluster composition, representative individual heatmaps, aggregate cluster heatmaps, and time series summaries.

In [ ]:
# Identify clusters in the data
n_clusters = len(x_array.cluster_photo.unique())
print(f"Number of clusters found: {n_clusters}")
print(f"Cluster labels: {sorted(x_array.cluster_photo.unique())}")

# Verify data has cluster information
print(f"\nCluster distribution:")
print(x_array['cluster_photo'].value_counts().sort_index())

### 3A. Pie Charts — Cluster Composition by Condition

In [ ]:
# Create pie charts for cluster composition across conditions
f, ax = plt.subplots(ncols=n_clusters, figsize=(1*n_clusters, 1.2),
                     gridspec_kw={'left': 0.05, 'right': 0.95, 'wspace': 0.01})

if n_clusters == 1:
    ax = [ax]  # Make iterable for consistency

all_clusters_observed_counts = []

for cluster_idx, cluster in enumerate(sorted(x_array.cluster_photo.unique())):
    tmp = x_array.query("cluster_photo == @cluster")
    total = len(tmp)
    
    observed_counts = [
        len(tmp.query("condition == 'replete' & infusiontype == '10NaCl'")),
        len(tmp.query("condition == 'replete' & infusiontype == '45NaCl'")),
        len(tmp.query("condition == 'deplete' & infusiontype == '10NaCl'")),
        len(tmp.query("condition == 'deplete' & infusiontype == '45NaCl'"))
    ]
    all_clusters_observed_counts.append(observed_counts)
    
    pie_props = [count / total if total > 0 else 0 for count in observed_counts]
    
    if total > 0:
        expected_counts = [total / 4, total / 4, total / 4, total / 4]
        chi2_stat, p_value = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)
        print(f"Cluster {cluster}: n={total}, Chi-squared p-value: {p_value:.4f}")
        significance_marker = "*" if p_value < 0.05 else ""
    else:
        significance_marker = ""
    
    ax[cluster_idx].pie(pie_props,
                  colors=colors,
                  explode=(0.1, 0.1, 0.1, 0.1),
                  )
    
    # ax[cluster_idx].text(0, -1.7, f"n={total}{significance_marker}", ha="center", va="center", fontsize=10, color="k")
    # ax[cluster_idx].set_title(f"Cluster {cluster_idx+1}", fontsize=11)

plt.tight_layout()
if SAVE_FIGS:
    save_figure_atomic(f, "figS5_pie_clusters", FIGSFOLDER)
plt.show()

# Save contingency table
contingency_table = np.array(all_clusters_observed_counts)
np.savetxt(RESULTSFOLDER / "cluster_contingency_table.csv", contingency_table, delimiter=",", fmt='%d')

# Overall Chi-squared test
if contingency_table.sum() > 0 and not (np.any(contingency_table.sum(axis=1) == 0) or np.any(contingency_table.sum(axis=0) == 0)):
    chi2_overall, p_overall, dof_overall, expected_freq_overall = stats.chi2_contingency(contingency_table)
    print("\nOverall Chi-squared Test for Independence:")
    print(f"Chi2 Statistic: {chi2_overall:.4f}, P-value: {p_overall:.4f}, DOF: {dof_overall}")

### 3B. Legend Patches — Condition Colors

In [ ]:
# Legend for replete conditions
f, ax = plt.subplots(figsize=(0.35, 0.45), gridspec_kw={'left': 0.01, 'right': 0.99})

legend_patches = []
legend_labels = ["10NaCl Replete", "45NaCl Replete"]
legend_labels = ["", ""]
for color, label in zip(colors[:2], legend_labels):
    patch = Patch(color=color, label=label)
    legend_patches.append(patch)

ax.legend(handles=legend_patches, loc='center', fontsize=8, frameon=False,
          labelspacing=0.2, borderpad=0.05)
ax.axis('off')

if SAVE_FIGS:
    save_figure_atomic(f, "figS5_legend_replete", FIGSFOLDER)
plt.show()

In [ ]:
# Legend for deplete conditions
f, ax = plt.subplots(figsize=(0.35, 0.45), gridspec_kw={'left': 0.01, 'right': 0.99})

legend_patches = []
legend_labels = ["10NaCl Deplete", "45NaCl Deplete"]
legend_labels = ["", ""]
for color, label in zip(colors[2:], legend_labels):
    patch = Patch(color=color, label=label)
    legend_patches.append(patch)

ax.legend(handles=legend_patches, loc='center', fontsize=8, frameon=False,
          labelspacing=0.2, borderpad=0.05)
ax.axis('off')

if SAVE_FIGS:
    save_figure_atomic(f, "figS5_legend_deplete", FIGSFOLDER)
plt.show()

### 3C. Heatmap — Clustered Dopamine Responses

In [ ]:
# Create clustered heatmap sorted by infusion response (AUC 5-15s window)
list_of_clustered_snips = []
n_of_clusters_list = []

for cluster in sorted(x_array.cluster_photo.unique()):
    # Get snips for this cluster
    cluster_mask = x_array.cluster_photo == cluster
    snips_cluster = snips_photo[cluster_mask, :]
    
    # Calculate AUC using trapezoidal rule (bins 50-150 = 5-15s)
    # Matches assembly script calculation: np.trapz() for true area under curve
    auc_infusion = np.array([np.trapezoid(snips_cluster[i, 50:150]) for i in range(len(snips_cluster))])
    
    # Verify against x_array auc_snips
    x_array_auc = x_array[cluster_mask]["auc_snips"].values
    auc_match = np.allclose(auc_infusion, x_array_auc, rtol=1e-5)
    
    print(f"Cluster {cluster}: {snips_cluster.shape[0]} trials")
    print(f"  Calculated AUC range: [{auc_infusion.min():.4f}, {auc_infusion.max():.4f}]")
    print(f"  x_array AUC range:    [{x_array_auc.min():.4f}, {x_array_auc.max():.4f}]")
    print(f"  AUC match verification: {auc_match} ✓" if auc_match else f"  AUC match verification: {auc_match} ✗ (mismatch!)")
    
    # Sort by AUC (descending - strongest response first)
    sort_order = np.argsort(auc_infusion)[::-1]
    snips_cluster_sorted = snips_cluster[sort_order, :]
    
    list_of_clustered_snips.append(snips_cluster_sorted)
    n_of_clusters_list.append(snips_cluster_sorted.shape[0])

clustered_snips = np.vstack(list_of_clustered_snips)

# Calculate dynamic vlim based on data
vlim = scale_vlim_to_data(clustered_snips, percentile=95)
# vlim = (-2, 2)  # Uncomment to override auto scaling with hardcoded values

# Create heatmap with cluster dividing lines
layout = [["ax", "cbar_ax"],
          ["ax", "empty"]]

f = plt.figure(figsize=(2.3, 3.6))
gs = f.add_gridspec(2, 2, width_ratios=[10, 1], height_ratios=[1, 1],
                     left=0.1, right=0.9, wspace=0.05, hspace=0.02)

ax = f.add_subplot(gs[:, 0])
cbar_ax = f.add_subplot(gs[0, 1])

sns.heatmap(clustered_snips, ax=ax, vmin=vlim[0], vmax=vlim[1], cmap="coolwarm", cbar_ax=cbar_ax)

ax.set_yticks([])
ax.set_xticks([])

cbar_ax.set_yticks([-2, 0, 2], labels=["-2", "0 Z", "2"])

# draw white line between clusters
cumulative_cluster_sizes = np.cumsum(n_of_clusters_list)
for cluster_boundary in cumulative_cluster_sizes[:-1]:  # Skip last boundary (end of heatmap)
    ax.axhline(cluster_boundary, color='white', linewidth=3)

if SAVE_FIGS:
    save_figure_atomic(f, "figS5_heatmap_clusters", FIGSFOLDER)
    


### 3D. Line Plots — Cluster Response Summary

In [ ]:
# Create summary line plots for each cluster
cluster_colors = ["black", "grey"]

f, [ax, blank] = plt.subplots(figsize=(2.3, 2.3), ncols=2,
                     gridspec_kw={'left': 0.1, 'right': 0.9, 'wspace': 0.05, 'hspace': 0.02,
                                  'width_ratios': [10, 1]})

blank.axis('off')  # Hide the blank subplot

for cluster_idx, cluster in enumerate(sorted(x_array.cluster_photo.unique())):
    color = cluster_colors[cluster_idx] if cluster_idx < len(cluster_colors) else "black"
    
    snips_cluster = snips_photo[x_array.cluster_photo == cluster, :]
    x = np.arange(snips_cluster.shape[1]) / 10  # Convert to seconds
    mean = np.mean(snips_cluster, axis=0)
    sd = np.std(snips_cluster, axis=0)
    sem = sd / np.sqrt(snips_cluster.shape[0])
    ci = sem * 1.96
    
    ax.plot(x, mean, color=color, lw=1.5, label=f"Cluster {cluster_idx+1} (n={snips_cluster.shape[0]})")
    ax.fill_between(x, mean-ci, mean+ci, alpha=0.1, color=color)

ax.set_xlim(0, 20)
# ax.axvline(5, color="k", linestyle="--", alpha=0.3, linewidth=0.8)
# ax.axvline(15, color="k", linestyle="--", alpha=0.3, linewidth=0.8)
ax.axvspan(5, 15, color="k", alpha=0.05, zorder=-20)
ax.set_xticks([])
ax.set_yticks([])
sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)

# Add scale bars
ax.plot([17, 17], [1, 2], color="k", linewidth=1)
ax.text(18, 1.5, "1 Z", ha="left", va="center", fontsize=9)

ax.plot([15, 20], [-0.8, -0.8], color="k", linewidth=1)
ax.text(17.5, -0.95, "5 s", ha="center", va="top", fontsize=9)

if SAVE_FIGS:
    save_figure_atomic(f, "figS5_line_clusters", FIGSFOLDER)
plt.show()

### 3E. Representative Heatmaps — Individual Animal Responses

In [ ]:
# Select exemplary animals for each cluster
vlim = scale_vlim_to_data(snips_photo, percentile=99)
# vlim = (-2, 2)  # Uncomment to override auto scaling with hardcoded values

# Find animals with most trials for representation
animal_trial_counts = x_array.groupby('id').size().sort_values(ascending=False)
exemplary_animals = animal_trial_counts.head(4).index.tolist()  # Get top 4 animals

print(f"Exemplary animals for figure: {exemplary_animals}")

for animal_id in exemplary_animals:
    animal_data = x_array[x_array.id == animal_id]
    
    f, ax = plt.subplots(ncols=2, figsize=(2, 1.4))
    
    replete_data = snips_photo[(x_array.id == animal_id) & (x_array.condition == "replete"), :]
    deplete_data = snips_photo[(x_array.id == animal_id) & (x_array.condition == "deplete"), :]
    
    if len(replete_data) > 0:
        sns.heatmap(data=replete_data, vmin=vlim[0], vmax=vlim[1], ax=ax[0], cmap="coolwarm", cbar=False)
        ax[0].set_title(f"{animal_id} Replete", fontsize=9)
    
    if len(deplete_data) > 0:
        sns.heatmap(data=deplete_data, vmin=vlim[0], vmax=vlim[1], ax=ax[1], cmap="coolwarm", cbar=False)
        ax[1].set_title(f"{animal_id} Deplete", fontsize=9)

## Organization Notes

This notebook generates Figure 3 components from the cluster analysis:
- **3A**: Pie charts showing the distribution of trials across conditions within each cluster
- **3B**: Legend patches for color identification
- **3C**: Heatmap of all trials organized by cluster, sorted by response magnitude
- **3D**: Summary line plots showing mean response ± 95% CI for each cluster
- **3E**: Representative individual animal heatmaps showing replete vs deplete responses

Set `SAVE_FIGS=True` in `figure_config.py` to export figures as PDF and PNG.

In [ ]:
print(f"Figure export status: SAVE_FIGS = {SAVE_FIGS}")
print(f"Output folder: {FIGSFOLDER}")
print(f"\nFigure 3 generation complete!")

## Exploratory: 4-Cluster Solution

Re-run the same PCA + SpectralClustering pipeline forced to k=4 clusters (vs. the k=2 solution in the main figures). This section is exploratory — it lets you inspect how the data would look if a finer-grained cluster structure is assumed.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score

N_CLUSTERS_EXPLORE = 4   # change to try other values
NUM_PCS = 3              # same as assembly script

# ── PCA (same settings as assemble_all_data.py) ──────────────────────
pca_exp = PCA(n_components=snips_photo.shape[1], whiten=True)
pca_exp.fit(snips_photo)
pca_transformed = pca_exp.transform(snips_photo)

# ── SpectralClustering ────────────────────────────────────────────────
sc_exp = SpectralClustering(
    n_clusters=N_CLUSTERS_EXPLORE,
    affinity="sigmoid",
    assign_labels="discretize",
    random_state=123,
)
sc_exp.fit(pca_transformed[:, :NUM_PCS])

sil_exp = silhouette_score(pca_transformed[:, :NUM_PCS], sc_exp.labels_, metric="cosine")
print(f"k={N_CLUSTERS_EXPLORE}  silhouette (cosine) = {sil_exp:.3f}")

# ── Reorder clusters so 0 = strongest positive infusion response ──────
pre_window = 50   # 5 s baseline × 10 Hz
uniquelabels_exp = list(set(sc_exp.labels_))
responses_exp = np.array([
    np.mean(snips_photo[sc_exp.labels_ == lbl, pre_window : 2 * pre_window])
    for lbl in uniquelabels_exp
])
rank = np.argsort(responses_exp)[::-1].astype(int)
rank_inv = np.array([np.where(rank == a)[0][0] for a in uniquelabels_exp])
cluster_photo_4 = np.array([rank_inv[np.digitize(l, uniquelabels_exp) - 1] for l in sc_exp.labels_])

x_array_4 = x_array.copy()
x_array_4["cluster_photo"] = cluster_photo_4

print("Cluster sizes:")
for cl, cnt in zip(*np.unique(cluster_photo_4, return_counts=True)):
    print(f"  Cluster {cl+1}: n={cnt}")

### E0. Silhouette sweep — compare k=2…8 to contextualise the 4-cluster solution

In [ ]:
# Run PCA once (shared across the sweep)
pca_sweep = PCA(n_components=snips_photo.shape[1], whiten=True)
pca_sweep.fit(snips_photo)
pca_tr_sweep = pca_sweep.transform(snips_photo)[:, :NUM_PCS]

k_values = range(2, 9)
sil_sweep = []

for k in k_values:
    sc_k = SpectralClustering(
        n_clusters=k, affinity="sigmoid",
        assign_labels="discretize", random_state=123,
    )
    sc_k.fit(pca_tr_sweep)
    sil_sweep.append(silhouette_score(pca_tr_sweep, sc_k.labels_, metric="cosine"))
    print(f"k={k}  silhouette={sil_sweep[-1]:.4f}")

f_sil, ax_sil = plt.subplots(figsize=(3.5, 2.2), gridspec_kw={"left": 0.15, "right": 0.95})
ax_sil.plot(list(k_values), sil_sweep, marker="o", color="steelblue", lw=1.5)
ax_sil.axvline(N_CLUSTERS_EXPLORE, color="tomato", ls="--", lw=1, label=f"k={N_CLUSTERS_EXPLORE} (explored)")
ax_sil.axvline(2, color="grey", ls=":", lw=1, label="k=2 (used in paper)")
ax_sil.set_xlabel("Number of clusters (k)")
ax_sil.set_ylabel("Silhouette score (cosine)")
ax_sil.set_xticks(list(k_values))
ax_sil.legend(fontsize=8, frameon=False)
ax_sil.set_title("Silhouette sweep")
sns.despine(ax=ax_sil)
plt.tight_layout()
plt.show()

### E1. Line Plots — Mean ± 95 % CI per cluster (k=4)

In [ ]:
cluster_palette_4 = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]  # up to 4 distinct colours

f, ax = plt.subplots(figsize=(3.5, 2.5), gridspec_kw={"left": 0.12, "right": 0.95})

for cl_idx, cl in enumerate(sorted(x_array_4.cluster_photo.unique())):
    mask = x_array_4.cluster_photo == cl
    snips_cl = snips_photo[mask.values, :]
    t = np.arange(snips_cl.shape[1]) / 10  # seconds

    mn = np.mean(snips_cl, axis=0)
    ci_4 = 1.96 * np.std(snips_cl, axis=0) / np.sqrt(snips_cl.shape[0])
    col = cluster_palette_4[cl_idx % len(cluster_palette_4)]

    ax.plot(t, mn, color=col, lw=1.5, label=f"Cluster {cl_idx+1} (n={snips_cl.shape[0]})")
    ax.fill_between(t, mn - ci_4, mn + ci_4, alpha=0.15, color=col)

ax.axvline(5,  color="k", ls="--", lw=0.8, alpha=0.4)
ax.axvline(15, color="k", ls="--", lw=0.8, alpha=0.4)
ax.set_xlim(0, 20)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Z-score")
ax.legend(fontsize=8, frameon=False)
ax.set_title(f"k={N_CLUSTERS_EXPLORE} spectral clustering")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

### E2. Heatmap — All trials sorted by cluster then by infusion AUC (k=4)

In [ ]:
list_4 = []
n_per_cluster_4 = []

for cl in sorted(x_array_4.cluster_photo.unique()):
    mask = x_array_4.cluster_photo == cl
    snips_cl = snips_photo[mask.values, :]
    auc_cl = np.array([np.trapezoid(snips_cl[i, 50:150]) for i in range(len(snips_cl))])
    order = np.argsort(auc_cl)[::-1]
    list_4.append(snips_cl[order, :])
    n_per_cluster_4.append(len(snips_cl))

all_snips_4 = np.vstack(list_4)
vlim_4 = scale_vlim_to_data(all_snips_4, percentile=95)

fig_h, (ax_h, cbar_ax_h) = plt.subplots(
    1, 2,
    figsize=(3.2, 4.5),
    gridspec_kw={"width_ratios": [20, 1], "left": 0.08, "right": 0.92, "wspace": 0.05},
)

sns.heatmap(
    all_snips_4, ax=ax_h, cbar_ax=cbar_ax_h,
    vmin=vlim_4[0], vmax=vlim_4[1], cmap="coolwarm",
)

# Draw dividing lines between clusters and add cluster labels
cumulative = 0
for cl_idx, n_cl in enumerate(n_per_cluster_4):
    if cl_idx > 0:
        ax_h.axhline(cumulative, color="white", lw=1.5)
    mid = cumulative + n_cl / 2
    ax_h.text(
        -2, mid,
        f"C{cl_idx+1}\n(n={n_cl})",
        va="center", ha="right", fontsize=7,
        color=cluster_palette_4[cl_idx % len(cluster_palette_4)],
    )
    cumulative += n_cl

# x-axis: mark infusion window (bins 50–150 = 5–15 s)
n_bins = all_snips_4.shape[1]
ax_h.set_xticks([0, 50, 150, n_bins])
ax_h.set_xticklabels(["0", "5", "15", f"{n_bins//10}"], fontsize=7)
ax_h.set_xlabel("Time (s)", fontsize=8)
ax_h.set_yticks([])
ax_h.set_title(f"k={N_CLUSTERS_EXPLORE} heatmap", fontsize=9)

plt.tight_layout()
plt.show()

### E3. Pie Charts — Condition composition per cluster (k=4)

In [ ]:
clusters_4_sorted = sorted(x_array_4.cluster_photo.unique())
f_pie, ax_pie = plt.subplots(
    1, len(clusters_4_sorted),
    figsize=(1.8 * len(clusters_4_sorted), 1.5),
    gridspec_kw={"left": 0.02, "right": 0.98},
)

obs_4 = []
for cl_idx, cl in enumerate(clusters_4_sorted):
    tmp4 = x_array_4.query("cluster_photo == @cl")
    total4 = len(tmp4)

    obs = [
        len(tmp4.query("condition == 'replete' & infusiontype == '10NaCl'")),
        len(tmp4.query("condition == 'replete' & infusiontype == '45NaCl'")),
        len(tmp4.query("condition == 'deplete' & infusiontype == '10NaCl'")),
        len(tmp4.query("condition == 'deplete' & infusiontype == '45NaCl'")),
    ]
    obs_4.append(obs)

    props = [c / total4 if total4 > 0 else 0 for c in obs]
    chi2_4, p4 = stats.chisquare(f_obs=obs, f_exp=[total4 / 4] * 4)
    sig4 = "*" if p4 < 0.05 else ""
    print(f"Cluster {cl_idx+1}: n={total4}  χ²={chi2_4:.2f}  p={p4:.4f}{sig4}")

    ax_pie[cl_idx].pie(
        props,
        colors=colors,
        explode=(0.08,) * 4,
    )
    ax_pie[cl_idx].set_title(f"C{cl_idx+1}\nn={total4}{sig4}", fontsize=9)

# Overall chi-squared across all 4 clusters
ct4 = np.array(obs_4)
if ct4.sum() > 0 and not (np.any(ct4.sum(axis=1) == 0) or np.any(ct4.sum(axis=0) == 0)):
    chi2_4all, p4all, dof4all, _ = stats.chi2_contingency(ct4)
    print(f"\nOverall χ²={chi2_4all:.4f}  p={p4all:.4f}  dof={dof4all}")

plt.suptitle("Condition composition (k=4)", y=1.02, fontsize=10)
plt.tight_layout()
plt.show()